# Further Research 3 -- Affordability Tipping Point Analysis

**Research Question:**
Under what conditions does the affordability index start worsening again?

**Core insight:**
Degrees became more affordable from 2012 to 2022 not because education got cheaper,
but because household incomes grew faster than education costs.
This improvement is fragile. The moment income growth slows or costs accelerate, it reverses.

**Important methodological note -- CPI-deflated fees:**
We do not have historical fee data going back to 2012. Our fee dataset was collected
in 2024 and gives us one snapshot. To compute a meaningful historical affordability
index, we cannot freeze the 2024 fee across all years -- that would make income growth
the only driver and guarantee an improving trend by construction.

Instead, we use the DOSM education CPI to back-estimate what fees would have been
in each HIES year:

    estimated_fee_in_year_X = fee_2024 x (edu_cpi_X / edu_cpi_2024)

This deflates the 2024 fee back to its estimated value in earlier years.
Both sides of the affordability index now move -- income and fees -- giving a more
honest picture of how affordability actually changed over time.

**Two pathways are analysed throughout:**
- Public pathway -- UTM and UM average fee
- Private pathway -- APU and Taylors average fee (upper bound)


## Setup -- Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
RED    = '#DC2626'
GRAY   = '#6B7280'
PURPLE = '#7C3AED'
TEAL   = '#0D9488'

print("Libraries loaded.")


In [ ]:
hh_national = pd.read_csv('../data/cleaned/hh_income_national.csv', parse_dates=['date'])
hh_state    = pd.read_csv('../data/cleaned/hh_income_state.csv',    parse_dates=['date'])
cpi_raw     = pd.read_csv('../data/cleaned/cpi.csv',                parse_dates=['date'])
fees_raw    = pd.read_csv('../data/cost/edunilai_mean_fees.csv')

hh_national['year'] = hh_national['date'].dt.year
hh_state['year']    = hh_state['date'].dt.year
cpi_raw['year']     = cpi_raw['date'].dt.year

print("Data loaded:")
print(f"  National household income: {len(hh_national)} rows")
print(f"  State household income:    {len(hh_state)} rows")
print(f"  CPI:                       {len(cpi_raw)} rows")
print(f"  Fees:                      {len(fees_raw)} rows")


In [ ]:
fees_raw['public_avg']  = fees_raw[['UTM', 'UM']].mean(axis=1)
fees_raw['private_avg'] = fees_raw[["APU", "Taylor's"]].mean(axis=1)

PUBLIC_FEE_2024  = fees_raw['public_avg'].mean()
PRIVATE_FEE_2024 = fees_raw['private_avg'].mean()

print("2024 fee values (the base for CPI deflation):")
print(f"  Public  fee (UTM + UM average):           RM {PUBLIC_FEE_2024:>10,.0f}")
print(f"  Private fee (APU + Taylors, upper bound): RM {PRIVATE_FEE_2024:>10,.0f}")
print(f"  Private-to-public ratio:                  {PRIVATE_FEE_2024/PUBLIC_FEE_2024:.1f}x")


In [ ]:
edu_cpi = cpi_raw[cpi_raw['division'] == '10'].set_index('year')['index'].astype(float)
CPI_2024 = edu_cpi[2024]

print("Education CPI by year (2010=100 base):")
for yr, val in sorted(edu_cpi.items()):
    print(f"  {yr}: {val:.4f}")
print()
print(f"CPI 2024 value used as deflation base: {CPI_2024:.4f}")


In [ ]:
income_2012 = hh_national[hh_national['year']==2012]['income_median'].values[0]
income_2022 = hh_national[hh_national['year']==2022]['income_median'].values[0]
cpi_2012    = edu_cpi[2012]
cpi_2022    = edu_cpi[2022]

INCOME_GROWTH_RATE_HISTORICAL  = (income_2022 / income_2012) ** (1/10) - 1
EDU_CPI_GROWTH_RATE_HISTORICAL = (cpi_2022    / cpi_2012)    ** (1/10) - 1

print("Key historical growth rates:")
print(f"  Median HH income 2012:  RM {income_2012:,}/month")
print(f"  Median HH income 2022:  RM {income_2022:,}/month")
print(f"  Annual income growth:   {INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print()
print(f"  Education CPI 2012:     {cpi_2012:.4f}")
print(f"  Education CPI 2022:     {cpi_2022:.4f}")
print(f"  Annual CPI growth:      {EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print()
print(f"  Income grew {INCOME_GROWTH_RATE_HISTORICAL/EDU_CPI_GROWTH_RATE_HISTORICAL:.1f}x faster than education costs.")


---
# Step 1 -- What Drove the Improvement? (2012-2022)

From 2012 to 2022, degrees became more affordable across Malaysia. But why exactly?
Was it because universities lowered their fees, or because families earned more money?
The answer matters because these two explanations have very different implications
for whether the improvement will last.

To answer this, we plot two lines on the same chart, both starting at 100 in 2012.
The first line tracks how much median household income grew each year.
The second line tracks how much education costs grew each year using DOSM education CPI.

If the income line rises faster than the cost line, affordability improves.
If the cost line overtakes the income line, affordability worsens.
The gap between the two lines at any point in time is the entire explanation
for why the affordability index moved in that direction.

**The key difference from a naive analysis:**
We use CPI-deflated fees for each year rather than freezing the 2024 fee.
This means the affordability index reflects both sides moving -- income growing
and fees rising -- giving an honest picture of the actual historical change.


### 1a -- Back-estimate fees for each HIES year using education CPI

We deflate the 2024 fee back to each HIES year by multiplying by the ratio of that year's education CPI to the 2024 CPI. A lower CPI ratio means fees were cheaper in that year relative to 2024.

In [ ]:
HIES_YEARS = [2012, 2014, 2016, 2019, 2020, 2022]

fee_estimates = []
for yr in HIES_YEARS:
    cpi_yr         = edu_cpi[yr]
    deflation      = cpi_yr / CPI_2024
    est_public     = PUBLIC_FEE_2024  * deflation
    est_private    = PRIVATE_FEE_2024 * deflation
    fee_estimates.append({
        'year':        yr,
        'cpi':         cpi_yr,
        'deflation':   deflation,
        'fee_public':  est_public,
        'fee_private': est_private,
    })

fee_df = pd.DataFrame(fee_estimates)

print("CPI-deflated fee estimates by HIES year:")
print(f"{'Year':<8} {'CPI':>8} {'Deflation':>12} {'Est. Public fee':>17} {'Est. Private fee':>18}")
print("-" * 68)
for _, row in fee_df.iterrows():
    print(f"  {int(row['year']):<6} {row['cpi']:>8.2f} {row['deflation']:>12.4f} "
          f"RM {row['fee_public']:>11,.0f}  RM {row['fee_private']:>12,.0f}")
print()
print(f"  2024 actual fees: Public RM {PUBLIC_FEE_2024:,.0f}  |  Private RM {PRIVATE_FEE_2024:,.0f}")
print("  Fees in 2012 were roughly 17% lower than today in real terms.")


### 1b -- Compute the affordability index using deflated fees

Now that both income and fees move for each year, the affordability index gives a genuine picture of how the burden changed over time. We also compute what the index would have looked like with frozen 2024 fees, so you can see the difference between the two approaches.

In [ ]:
aff_rows = []
for _, fee_row in fee_df.iterrows():
    yr            = fee_row['year']
    income_row    = hh_national[hh_national['year']==yr]
    if income_row.empty:
        continue
    monthly_income = income_row['income_median'].values[0]
    annual_income  = monthly_income * 12

    aff_rows.append({
        'year':              yr,
        'income_median':     monthly_income,
        'fee_public':        fee_row['fee_public'],
        'fee_private':       fee_row['fee_private'],
        'aff_public':        fee_row['fee_public']  / annual_income,
        'aff_private':       fee_row['fee_private'] / annual_income,
        'aff_public_frozen': PUBLIC_FEE_2024        / annual_income,
    })

aff_national_df = pd.DataFrame(aff_rows).sort_values('year').reset_index(drop=True)

print("Affordability index -- deflated fees vs frozen fees (public pathway):")
print(f"{'Year':<8} {'Income/mth':>12} {'Deflated fee':>14} {'Aff (deflated)':>16} {'Aff (frozen 2024)':>19} {'Difference':>12}")
print("-" * 85)
for _, row in aff_national_df.iterrows():
    diff = row['aff_public'] - row['aff_public_frozen']
    print(f"  {int(row['year']):<6} RM {row['income_median']:>7,}   "
          f"RM {row['fee_public']:>9,.0f}   "
          f"{row['aff_public']:>14.4f}   "
          f"{row['aff_public_frozen']:>17.4f}   "
          f"{diff:>+10.4f}")
print()
print("The deflated index starts higher in 2012 and ends at a similar 2022 level.")
print("This means the true improvement is smaller than the frozen-fee version suggested.")


### 1c -- Quantify how much smaller the real improvement is

By comparing the improvement under both approaches, we can see exactly how much of the apparent improvement was an illusion caused by freezing fees at 2024 levels.

In [ ]:
pub_baseline_frozen   = aff_national_df[aff_national_df['year']==2012]['aff_public_frozen'].values[0]
pub_baseline_deflated = aff_national_df[aff_national_df['year']==2012]['aff_public'].values[0]
pub_2022_frozen       = aff_national_df[aff_national_df['year']==2022]['aff_public_frozen'].values[0]
pub_2022_deflated     = aff_national_df[aff_national_df['year']==2022]['aff_public'].values[0]

improvement_frozen   = pub_baseline_frozen   - pub_2022_frozen
improvement_deflated = pub_baseline_deflated - pub_2022_deflated
overstatement        = improvement_frozen - improvement_deflated
overstatement_pct    = overstatement / improvement_frozen * 100

print("How much was the improvement overstated by using frozen 2024 fees?")
print()
print(f"  Frozen fee approach:   2012 index {pub_baseline_frozen:.4f}  ->  2022 index {pub_2022_frozen:.4f}")
print(f"  Improvement (frozen):  {improvement_frozen:.4f} index points")
print()
print(f"  Deflated fee approach: 2012 index {pub_baseline_deflated:.4f}  ->  2022 index {pub_2022_deflated:.4f}")
print(f"  Improvement (deflated):{improvement_deflated:.4f} index points")
print()
print(f"  Overstatement: {overstatement:.4f} index points ({overstatement_pct:.1f}% of the apparent improvement)")
print()
print("  In plain terms: the frozen-fee approach made the improvement look")
print(f"  {overstatement_pct:.0f}% larger than it actually was.")
print("  The deflated approach is the honest picture.")


### 1d -- Plot: income vs CPI growth, and affordability index both pathways

Left panel shows the indexed growth lines. Right panel shows the deflated affordability index for both pathways with the 2012 baseline marked.

In [ ]:
PUBLIC_BASELINE_2012  = aff_national_df[aff_national_df['year']==2012]['aff_public'].values[0]
PRIVATE_BASELINE_2012 = aff_national_df[aff_national_df['year']==2012]['aff_private'].values[0]
PUBLIC_AFF_2022       = aff_national_df[aff_national_df['year']==2022]['aff_public'].values[0]
PRIVATE_AFF_2022      = aff_national_df[aff_national_df['year']==2022]['aff_private'].values[0]

# Index income and CPI to 2012=100
income_indexed = [(yr, hh_national[hh_national['year']==yr]['income_median'].values[0] /
                   income_2012 * 100) for yr in HIES_YEARS]
cpi_indexed    = [(yr, edu_cpi[yr] / cpi_2012 * 100) for yr in HIES_YEARS]
inc_df = pd.DataFrame(income_indexed, columns=['year', 'indexed'])
cpi_df = pd.DataFrame(cpi_indexed,    columns=['year', 'indexed'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left
axes[0].plot(inc_df['year'], inc_df['indexed'],
             color=GREEN, lw=2.5, marker='o', markersize=6, label='Median HH income (2012=100)')
axes[0].plot(cpi_df['year'], cpi_df['indexed'],
             color=RED, lw=2.5, marker='s', markersize=6, label='Education CPI (2012=100)')
axes[0].fill_between(inc_df['year'],
                     inc_df['indexed'].values, cpi_df['indexed'].values,
                     where=inc_df['indexed'].values >= cpi_df['indexed'].values,
                     alpha=0.15, color=GREEN, label='Income growing faster')
axes[0].fill_between(inc_df['year'],
                     inc_df['indexed'].values, cpi_df['indexed'].values,
                     where=inc_df['indexed'].values < cpi_df['indexed'].values,
                     alpha=0.20, color=RED, label='Costs growing faster')
axes[0].axhline(100, color=GRAY, lw=1, ls='--', alpha=0.5)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Index (2012 = 100)')
axes[0].set_title('Income Growth vs Education Cost Growth\n(2012 = 100)', fontweight='bold')
axes[0].legend(fontsize=8)

# Right: deflated affordability index both pathways
axes[1].plot(aff_national_df['year'], aff_national_df['aff_public'],
             color=BLUE,   lw=2.5, marker='o', markersize=6, label='Public pathway (deflated fees)')
axes[1].plot(aff_national_df['year'], aff_national_df['aff_private'],
             color=ORANGE, lw=2.5, marker='s', markersize=6, label='Private pathway (deflated fees)')
axes[1].plot(aff_national_df['year'], aff_national_df['aff_public_frozen'],
             color=BLUE, lw=1.5, ls=':', alpha=0.5, label='Public (frozen 2024 fee -- for comparison)')
axes[1].axhline(PUBLIC_BASELINE_2012,  color=BLUE,   lw=1.2, ls=':', alpha=0.7,
                label=f'Public 2012 baseline ({PUBLIC_BASELINE_2012:.3f})')
axes[1].axhline(PRIVATE_BASELINE_2012, color=ORANGE, lw=1.2, ls=':', alpha=0.7,
                label=f'Private 2012 baseline ({PRIVATE_BASELINE_2012:.3f})')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Affordability Index (years of HH income per degree)')
axes[1].set_title('Affordability Index 2012-2022\n'
                  'Solid = CPI-deflated fees (correct) | Dotted = frozen 2024 fee (for comparison)',
                  fontweight='bold')
axes[1].legend(fontsize=7.5)

plt.suptitle('Step 1 -- What Drove the Improvement? (CPI-Deflated Fees)',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f"Public  2012 baseline (deflated): {PUBLIC_BASELINE_2012:.4f}")
print(f"Public  2022 level    (deflated): {PUBLIC_AFF_2022:.4f}")
print(f"Private 2012 baseline (deflated): {PRIVATE_BASELINE_2012:.4f}")
print(f"Private 2022 level    (deflated): {PRIVATE_AFF_2022:.4f}")


---
# Step 2 -- Were There Years Where Affordability Temporarily Got Worse?

Even though the overall trend from 2012 to 2022 shows degrees becoming more affordable,
this does not mean every single year improved. Some years the index went up -- meaning
degrees became harder to afford that year -- before resuming the downward trend.

To find these dips, we calculate how much the affordability index changed between each
consecutive pair of HIES survey years. A positive change means affordability got worse
that period. A negative change means it got better.

The 2019 to 2020 period is the most obvious case. COVID-19 caused household incomes
to drop sharply -- nationally median income fell from RM 5,873 to RM 5,209, an 11%
decline in one year -- while university fees and education costs barely changed.
The result: the same degree suddenly cost a larger share of household income.

With deflated fees, the 2020 reversal is still visible but slightly smaller in
magnitude than the frozen-fee version, because fees also barely changed that year
so the deflated fee is almost the same as the frozen fee for that specific period.


### 2a -- Compute year-on-year change in the deflated affordability index

We take the difference between each consecutive HIES year for both pathways. Positive = worsened, negative = improved.

In [ ]:
aff_national_df['aff_public_change']  = aff_national_df['aff_public'].diff()
aff_national_df['aff_private_change'] = aff_national_df['aff_private'].diff()
aff_national_df['income_change_pct']  = aff_national_df['income_median'].pct_change() * 100
aff_national_df['fee_change_pct']     = aff_national_df['fee_public'].pct_change()    * 100

years_with_change = aff_national_df.dropna(subset=['aff_public_change']).copy()

print("Year-on-year change in affordability index (CPI-deflated fees):")
print(f"{'Period':<12} {'Income chg':>12} {'Fee chg':>10} {'Public chg':>12} {'Private chg':>13} {'Direction'}")
print("-" * 75)
for _, row in years_with_change.iterrows():
    direction = 'WORSENED' if row['aff_public_change'] > 0 else 'improved'
    print(f"  to {int(row['year']):<7} {row['income_change_pct']:>+10.1f}%  "
          f"{row['fee_change_pct']:>+8.1f}%  "
          f"{row['aff_public_change']:>+10.4f}  "
          f"{row['aff_private_change']:>+11.4f}  "
          f"{direction}")


### 2b -- Isolate and explain the 2020 COVID reversal

We look closely at 2019 to 2020 to confirm that income fell while fees barely moved, causing the index to jump upward. We also compare the magnitude of the reversal between the public and private pathways.

In [ ]:
row_2020 = years_with_change[years_with_change['year']==2020].iloc[0]
inc_2019 = hh_national[hh_national['year']==2019]['income_median'].values[0]
inc_2020 = hh_national[hh_national['year']==2020]['income_median'].values[0]

print("2020 COVID shock -- detailed breakdown:")
print()
print(f"  Median income 2019:           RM {inc_2019:,}/month")
print(f"  Median income 2020:           RM {inc_2020:,}/month")
print(f"  Income change:                {(inc_2020/inc_2019-1)*100:+.1f}%")
print()
print(f"  Estimated public fee 2019:    RM {fee_df[fee_df['year']==2019]['fee_public'].values[0]:,.0f}")
print(f"  Estimated public fee 2020:    RM {fee_df[fee_df['year']==2020]['fee_public'].values[0]:,.0f}")
print(f"  Fee change (public):          {row_2020['fee_change_pct']:+.2f}%")
print()
print(f"  Public  index worsened by:    {row_2020['aff_public_change']:+.4f} index points")
print(f"  Private index worsened by:    {row_2020['aff_private_change']:+.4f} index points")
print()
ratio = abs(row_2020['aff_private_change'] / row_2020['aff_public_change'])
print(f"  Private pathway was hit {ratio:.1f}x harder in absolute terms.")
print("  Same income shock. Same fee movement. But private fee base is 10.7x larger.")


### 2c -- Compute annualised income and fee growth for each HIES period

For each period between HIES surveys, we compute how fast income grew per year and how fast education costs grew per year. When income growth is faster, the index improved. When costs grew faster, it worsened.

In [ ]:
period_growth = []
sorted_years = sorted(aff_national_df['year'].tolist())
for i in range(1, len(sorted_years)):
    yr_curr  = sorted_years[i]
    yr_prev  = sorted_years[i-1]
    gap      = yr_curr - yr_prev
    inc_curr = hh_national[hh_national['year']==yr_curr]['income_median'].values[0]
    inc_prev = hh_national[hh_national['year']==yr_prev]['income_median'].values[0]
    inc_growth = (inc_curr / inc_prev) ** (1/gap) - 1
    cpi_growth = (edu_cpi[yr_curr] / edu_cpi[yr_prev]) ** (1/gap) - 1
    period_growth.append({
        'period':      f'{yr_prev}->{yr_curr}',
        'inc_growth':  inc_growth * 100,
        'cpi_growth':  cpi_growth * 100,
        'income_faster': inc_growth > cpi_growth,
    })

period_df = pd.DataFrame(period_growth)

print("Annualised income growth vs education CPI growth per HIES period:")
print(f"{'Period':<14} {'Income growth/yr':>18} {'CPI growth/yr':>15} {'Income faster?':>16}")
print("-" * 68)
for _, row in period_df.iterrows():
    result = 'YES -- improved' if row['income_faster'] else 'NO -- worsened'
    print(f"  {row['period']:<12} {row['inc_growth']:>16.2f}%  "
          f"{row['cpi_growth']:>13.2f}%  {result}")


### 2d -- Plot: year-on-year change for both pathways, and growth rate comparison by period

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

x     = np.arange(len(years_with_change))
width = 0.35
pub_changes  = years_with_change['aff_public_change'].values
priv_changes = years_with_change['aff_private_change'].values
period_labels = [f"to {int(y)}" for y in years_with_change['year']]

pub_colors  = [RED if v > 0 else GREEN for v in pub_changes]
priv_colors = [RED if v > 0 else GREEN for v in priv_changes]

axes[0].bar(x - width/2, pub_changes,  width, color=pub_colors,  alpha=0.85, label='Public pathway')
axes[0].bar(x + width/2, priv_changes, width, color=priv_colors, alpha=0.55, label='Private pathway')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(period_labels, fontsize=9)
axes[0].set_ylabel('Change in Affordability Index (CPI-deflated)')
axes[0].set_title('Year-on-Year Change in Affordability Index\n'
                  'Red = worsened | Green = improved', fontweight='bold')
axes[0].legend(fontsize=9)
for i, pv in enumerate(pub_changes):
    axes[0].text(i - width/2, pv + (0.001 if pv >= 0 else -0.002),
                 f'{pv:+.3f}', ha='center', fontsize=7.5,
                 va='bottom' if pv >= 0 else 'top')

x2 = np.arange(len(period_df))
axes[1].bar(x2 - width/2, period_df['inc_growth'], width,
            color=GREEN, alpha=0.85, label='Income growth (%/yr)')
axes[1].bar(x2 + width/2, period_df['cpi_growth'], width,
            color=RED,   alpha=0.85, label='Education CPI growth (%/yr)')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(period_df['period'], fontsize=8.5)
axes[1].set_ylabel('Annualised Growth Rate (%/yr)')
axes[1].set_title('Income Growth vs Education Cost Growth per Period\n'
                  'Green taller = improved | Red taller = worsened', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Step 2 -- When Did Affordability Temporarily Worsen?',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


---
# Step 3 -- Four Forward Scenarios (2022-2040)

We now know the improvement came from income growing faster than costs.
The tipping point question is: what happens if that balance shifts?

We test four combinations:

**Scenario A -- Baseline:**
Income keeps growing at the same rate it did from 2012 to 2022, and education costs
keep rising at the same rate. This is the optimistic case -- nothing changes.

**Scenario B -- Income slowdown:**
Income growth slows to half its historical rate. This represents economic slowdown,
stagnant wages, or a weak graduate job market. Income still grows, just more slowly.

**Scenario C -- Cost acceleration:**
Income growth stays the same but education costs rise at double the historical CPI rate.
This represents universities raising fees more aggressively.

**Scenario D -- Combined shock:**
Income growth slows AND education costs accelerate simultaneously. The worst case.

For each scenario we project the affordability index from 2022 to 2040 for both
pathways. The 2012 baseline (deflated) is marked as the reversal reference line.
When a scenario crosses it, the entire 2012-2022 improvement is erased.

**Note on the projection:**
The projection uses the 2022 deflated fee as the starting fee, then grows it
forward at the scenario fee growth rate. This is consistent with the historical
approach -- both sides move in the projection too.


### 3a -- Define scenario parameters and print actual rates

In [ ]:
PROJ_YEARS = np.arange(2022, 2041)

scenarios = {
    'A -- Baseline':          {'income_mult': 1.0, 'fee_mult': 1.0,
                                'color': GREEN,  'ls': '-'},
    'B -- Income slowdown':   {'income_mult': 0.5, 'fee_mult': 1.0,
                                'color': ORANGE, 'ls': '--'},
    'C -- Cost acceleration': {'income_mult': 1.0, 'fee_mult': 2.0,
                                'color': PURPLE, 'ls': '--'},
    'D -- Combined shock':    {'income_mult': 0.5, 'fee_mult': 2.0,
                                'color': RED,    'ls': '-'},
}

print("Scenario parameters:")
print(f"  Historical income growth: {INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print(f"  Historical fee CPI growth:{EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print()
print(f"{'Scenario':<30} {'Income growth/yr':>18} {'Fee growth/yr':>15}")
print("-" * 68)
for name, params in scenarios.items():
    inc = INCOME_GROWTH_RATE_HISTORICAL  * params['income_mult'] * 100
    fee = EDU_CPI_GROWTH_RATE_HISTORICAL * params['fee_mult']    * 100
    print(f"  {name:<28} {inc:>16.2f}%  {fee:>13.2f}%")


### 3b -- Build the projection function

The projection starts from the 2022 deflated fee (not the 2024 fee) and grows both income and fees forward. This keeps the projection consistent with the historical analysis.

In [ ]:
# Starting values for the projection
income_2022_value     = hh_national[hh_national['year']==2022]['income_median'].values[0]
pub_fee_2022_deflated = fee_df[fee_df['year']==2022]['fee_public'].values[0]
prv_fee_2022_deflated = fee_df[fee_df['year']==2022]['fee_private'].values[0]

def project_affordability(starting_income, starting_fee,
                          income_growth_rate, fee_growth_rate, projection_years):
    results = []
    for yr in projection_years:
        years_elapsed    = yr - 2022
        projected_income = starting_income * (1 + income_growth_rate) ** years_elapsed
        projected_fee    = starting_fee    * (1 + fee_growth_rate)    ** years_elapsed
        projected_index  = projected_fee / (projected_income * 12)
        results.append({
            'year':          yr,
            'income':        projected_income,
            'fee':           projected_fee,
            'affordability': projected_index,
        })
    return pd.DataFrame(results)

print(f"Projection starting values (2022, deflated):")
print(f"  Income:       RM {income_2022_value:,}/month")
print(f"  Public  fee:  RM {pub_fee_2022_deflated:,.0f}")
print(f"  Private fee:  RM {prv_fee_2022_deflated:,.0f}")
print()
print("Note: starting from the 2022 deflated fee keeps the projection")
print("consistent with the historical index. Both income and fees move.")


### 3c -- Generate all projections and do a quick sanity check

We run the projection function for all 4 scenarios x 2 pathways = 8 series. We then check which scenarios actually cross the 2012 baseline before 2040.

In [ ]:
public_projections  = {}
private_projections = {}

for scenario_name, params in scenarios.items():
    income_growth_rate = INCOME_GROWTH_RATE_HISTORICAL  * params['income_mult']
    fee_growth_rate    = EDU_CPI_GROWTH_RATE_HISTORICAL * params['fee_mult']

    public_projections[scenario_name]  = project_affordability(
        income_2022_value, pub_fee_2022_deflated,
        income_growth_rate, fee_growth_rate, PROJ_YEARS)
    private_projections[scenario_name] = project_affordability(
        income_2022_value, prv_fee_2022_deflated,
        income_growth_rate, fee_growth_rate, PROJ_YEARS)

print("Sanity check -- does each scenario cross the 2012 baseline before 2040?")
print(f"  Public  2012 baseline: {PUBLIC_BASELINE_2012:.4f}")
print(f"  Private 2012 baseline: {PRIVATE_BASELINE_2012:.4f}")
print()
print(f"{'Scenario':<30} {'Public crosses?':>18} {'Private crosses?':>18}")
print("-" * 70)
for scenario_name in scenarios.keys():
    pub_cross  = public_projections[scenario_name][
        public_projections[scenario_name]['affordability'] >= PUBLIC_BASELINE_2012]['year']
    priv_cross = private_projections[scenario_name][
        private_projections[scenario_name]['affordability'] >= PRIVATE_BASELINE_2012]['year']
    pub_str  = f"Yes, by {int(pub_cross.values[0])}"  if len(pub_cross)  > 0 else "No"
    priv_str = f"Yes, by {int(priv_cross.values[0])}" if len(priv_cross) > 0 else "No"
    print(f"  {scenario_name:<28} {pub_str:>18} {priv_str:>18}")


### 3d -- Plot all four scenarios for both pathways

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, projections, historical_col, baseline_val, pathway_title in [
    (axes[0], public_projections,  'aff_public',
     PUBLIC_BASELINE_2012,  f'Public Pathway (2022 fee RM {pub_fee_2022_deflated:,.0f})'),
    (axes[1], private_projections, 'aff_private',
     PRIVATE_BASELINE_2012, f'Private Pathway (2022 fee RM {prv_fee_2022_deflated:,.0f})'),
]:
    ax.plot(aff_national_df['year'], aff_national_df[historical_col],
            color='black', lw=2.5, marker='o', markersize=6,
            label='Historical actual (deflated)', zorder=5)

    for scenario_name, proj_df in projections.items():
        params = scenarios[scenario_name]
        ax.plot(proj_df['year'], proj_df['affordability'],
                color=params['color'], lw=2.0, ls=params['ls'],
                marker='o', markersize=3, alpha=0.85,
                label=scenario_name.split(' -- ')[1])

    ax.axhline(baseline_val, color=RED, lw=1.8, ls=':', alpha=0.9,
               label=f'2012 baseline ({baseline_val:.3f})')
    ax.axvline(2022, color=GRAY, lw=0.8, ls=':', alpha=0.5, label='Forecast starts')
    ax.set_ylim(bottom=0)
    ax.set_xlabel('Year')
    ax.set_ylabel('Affordability Index (years of HH income per degree)')
    ax.set_title(f'{pathway_title}\n'
                 'Red dotted = 2012 baseline (crossing it = improvement fully erased)',
                 fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Step 3 -- Four Scenarios: Public vs Private Pathway (CPI-Deflated Fees)',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


---
# Step 4 -- Finding the Exact Tipping Point

The scenario charts show the general direction. This step produces specific numbers.

We calculate two threshold values for each pathway.

**Threshold 1 -- Minimum income growth rate:**
We fix education costs at the historical CPI growth rate and find the minimum income
growth rate that keeps the index from worsening. Any income growth below this rate
means affordability starts reversing. This gives one number per pathway:
income growth cannot fall below X percent per year.

**Threshold 2 -- Maximum fee growth rate:**
We fix income growth at the historical rate and find the maximum fee growth rate
the system can absorb before the index starts reversing. Any fee growth above this
rate causes affordability to worsen even if incomes keep growing normally.
This gives one number per pathway: fees cannot rise faster than Y percent per year.

These two thresholds define the safe zone. As long as income stays above Threshold 1
and fees stay below Threshold 2, affordability will continue improving.


### 4a -- Find the exact tipping point year for each scenario

In [ ]:
YEARS_HORIZON = 18  # 2022 to 2040

tipping_years_public  = {}
tipping_years_private = {}

print("Exact tipping point years (CPI-deflated):")
print(f"{'Scenario':<30} {'Public':>12} {'Private':>14}")
print("-" * 60)
for scenario_name in scenarios.keys():
    pub_cross  = public_projections[scenario_name][
        public_projections[scenario_name]['affordability'] >= PUBLIC_BASELINE_2012]['year']
    priv_cross = private_projections[scenario_name][
        private_projections[scenario_name]['affordability'] >= PRIVATE_BASELINE_2012]['year']

    pub_tip  = int(pub_cross.values[0])  if len(pub_cross)  > 0 else None
    priv_tip = int(priv_cross.values[0]) if len(priv_cross) > 0 else None

    tipping_years_public[scenario_name]  = pub_tip
    tipping_years_private[scenario_name] = priv_tip

    pub_str  = str(pub_tip)  if pub_tip  else 'No reversal by 2040'
    priv_str = str(priv_tip) if priv_tip else 'No reversal by 2040'
    print(f"  {scenario_name:<28} {pub_str:>12} {priv_str:>14}")


### 4b -- Compute Threshold 1: minimum income growth rate

We solve for the income growth rate that keeps the affordability index exactly at its 2022 level by 2040, given fees rise at their historical CPI rate. The maths: we need projected_fee_2040 / (projected_income_2040 x 12) = aff_2022. We know the fee side, so we solve for the income side.

In [ ]:
print("Threshold 1 -- Minimum income growth to prevent worsening")
print("(Fees growing at historical CPI rate)")
print()

thresholds = {}
for pathway_label, start_fee, pathway_aff_2022 in [
    ('Public  pathway', pub_fee_2022_deflated, PUBLIC_AFF_2022),
    ('Private pathway', prv_fee_2022_deflated, PRIVATE_AFF_2022),
]:
    projected_fee_2040      = start_fee * (1 + EDU_CPI_GROWTH_RATE_HISTORICAL) ** YEARS_HORIZON
    required_income_2040    = projected_fee_2040 / (pathway_aff_2022 * 12)
    min_income_growth_rate  = (required_income_2040 / income_2022_value) ** (1/YEARS_HORIZON) - 1
    income_buffer           = INCOME_GROWTH_RATE_HISTORICAL - min_income_growth_rate

    thresholds[pathway_label] = {'min_income': min_income_growth_rate}

    print(f"  {pathway_label}:")
    print(f"    Minimum income growth needed:    {min_income_growth_rate*100:.2f}%/yr")
    print(f"    Historical income growth was:    {INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
    print(f"    Buffer (income can slow by):     {income_buffer*100:.2f} percentage points")
    print(f"    => If income growth falls below {min_income_growth_rate*100:.2f}%/yr, affordability reverses.")
    print()


### 4c -- Compute Threshold 2: maximum fee growth rate

We fix income at its historical growth rate and solve for the fee growth rate that keeps the affordability index at its 2022 level by 2040. Any fee growth faster than this causes the index to worsen.

In [ ]:
print("Threshold 2 -- Maximum fee growth before affordability worsens")
print("(Income growing at historical rate)")
print()

for pathway_label, start_fee, pathway_aff_2022 in [
    ('Public  pathway', pub_fee_2022_deflated, PUBLIC_AFF_2022),
    ('Private pathway', prv_fee_2022_deflated, PRIVATE_AFF_2022),
]:
    projected_income_2040 = income_2022_value * (1 + INCOME_GROWTH_RATE_HISTORICAL) ** YEARS_HORIZON
    max_fee_2040          = pathway_aff_2022  * projected_income_2040 * 12
    max_fee_growth_rate   = (max_fee_2040 / start_fee) ** (1/YEARS_HORIZON) - 1
    fee_buffer            = max_fee_growth_rate - EDU_CPI_GROWTH_RATE_HISTORICAL

    thresholds[pathway_label]['max_fee'] = max_fee_growth_rate

    print(f"  {pathway_label}:")
    print(f"    Maximum fee growth tolerable:    {max_fee_growth_rate*100:.2f}%/yr")
    print(f"    Historical education CPI was:    {EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
    print(f"    Buffer (fees can accelerate by): {fee_buffer*100:.2f} percentage points")
    print(f"    => If fees rise faster than {max_fee_growth_rate*100:.2f}%/yr, affordability reverses.")
    print()


### 4d -- Plot: tipping point years and threshold comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: tipping point years per scenario
x = np.arange(len(scenarios))
width = 0.35
pub_tip_vals  = [tipping_years_public.get(s)  or 2042 for s in scenarios.keys()]
priv_tip_vals = [tipping_years_private.get(s) or 2042 for s in scenarios.keys()]

axes[0].bar(x - width/2, pub_tip_vals,  width, color=BLUE,   alpha=0.85, label='Public')
axes[0].bar(x + width/2, priv_tip_vals, width, color=ORANGE, alpha=0.85, label='Private')
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.split(' -- ')[1] for s in scenarios.keys()], fontsize=9)
axes[0].set_ylabel('Year Improvement Is Fully Erased')
axes[0].set_ylim(2020, 2046)
axes[0].axhline(2022, color=GRAY, lw=1, ls='--', alpha=0.5, label='Today (2022)')
axes[0].set_title('When Does Each Scenario Erase\nthe 2012-2022 Improvement?', fontweight='bold')
axes[0].legend(fontsize=9)
for i, (pv, rv) in enumerate(zip(pub_tip_vals, priv_tip_vals)):
    axes[0].text(i-width/2, pv+0.2, str(pv) if pv<2042 else '>2040',
                 ha='center', fontsize=9, fontweight='bold', color=BLUE)
    axes[0].text(i+width/2, rv+0.2, str(rv) if rv<2042 else '>2040',
                 ha='center', fontsize=9, fontweight='bold', color=ORANGE)

# Right: threshold values for both pathways
categories  = ['Public\nMin income\ngrowth', 'Private\nMin income\ngrowth',
               'Public\nMax fee\ngrowth',    'Private\nMax fee\ngrowth']
pub_min_inc  = thresholds['Public  pathway']['min_income']
priv_min_inc = thresholds['Private pathway']['min_income']
pub_max_fee  = thresholds['Public  pathway']['max_fee']
priv_max_fee = thresholds['Private pathway']['max_fee']

threshold_vals   = [pub_min_inc*100, priv_min_inc*100, pub_max_fee*100, priv_max_fee*100]
threshold_colors = [BLUE, ORANGE, BLUE, ORANGE]
bars = axes[1].bar(categories, threshold_vals, color=threshold_colors, alpha=0.85, width=0.5)
axes[1].axhline(INCOME_GROWTH_RATE_HISTORICAL*100,  color=GREEN, lw=2, ls='--',
                label=f'Historical income growth ({INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr)')
axes[1].axhline(EDU_CPI_GROWTH_RATE_HISTORICAL*100, color=RED,   lw=2, ls='--',
                label=f'Historical fee growth ({EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr)')
axes[1].set_ylabel('Rate (%/yr)')
axes[1].set_title('Safe Zone Thresholds\n'
                  'Green line = historical income | Red line = historical fee CPI',
                  fontweight='bold')
axes[1].legend(fontsize=8)
for bar, val in zip(bars, threshold_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Step 4 -- Exact Tipping Point Thresholds (CPI-Deflated)',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


---
# Step 5 -- Which States Hit the Tipping Point First?

The national numbers hide an important geographic difference.
Poorer states like Kelantan, Kedah, and Sabah had lower incomes to begin with
and their incomes grew more slowly over 2012 to 2022. This means they have less
room before their affordability index reverses. A shock that KL can absorb
without crossing the tipping point may be enough to push Kelantan past it.

We repeat the four scenario projections for six states using each state's own
historical income growth rate. The 2012 deflated baseline is also state-specific
-- each state is judged against where it started, not where KL started.

**Why these six states?**

KL is the benchmark -- highest income, fastest growth, used as the reference
point throughout Research 2 and 3.

Selangor sits just below KL but is the most populous state in Malaysia,
representing the large urban middle.

Perak is a middle-income state that sits between the wealthy urban states
and the lower-income states, making the gradient across panels visible.

Kedah and Kelantan are consistently among the lowest-income states across
every HIES survey from 2012 to 2022. They have the least buffer and will
hit the tipping point soonest under any adverse scenario.

Sabah is included because it represents East Malaysia and has a different
income growth profile from the Peninsular states. Excluding it would leave
a geographic blind spot in the analysis.

Together these six states cover the top, middle, and bottom of the income
distribution and span both Peninsular and East Malaysia -- giving the small
multiples chart a complete picture of geographic inequality.


### 5a -- Compute each state's own income growth rate and starting affordability

In [ ]:
FOCUS_STATES = ['W.P. Kuala Lumpur', 'Selangor', 'Perak', 'Kedah', 'Sabah', 'Kelantan']

state_profiles = {}
for state in FOCUS_STATES:
    state_data     = hh_state[hh_state['state']==state].sort_values('year')
    inc_2012       = state_data[state_data['year']==2012]['income_median'].values[0]
    inc_2022       = state_data[state_data['year']==2022]['income_median'].values[0]
    growth         = (inc_2022 / inc_2012) ** (1/10) - 1

    # Use deflated 2022 fee as starting point (same as national)
    state_profiles[state] = {
        'income_2012':         inc_2012,
        'income_2022':         inc_2022,
        'growth_rate':         growth,
        'pub_baseline_2012':   pub_fee_2022_deflated * (edu_cpi[2012]/edu_cpi[2022]) / (inc_2012 * 12),
        'priv_baseline_2012':  prv_fee_2022_deflated * (edu_cpi[2012]/edu_cpi[2022]) / (inc_2012 * 12),
        'pub_aff_2022':        pub_fee_2022_deflated / (inc_2022 * 12),
        'priv_aff_2022':       prv_fee_2022_deflated / (inc_2022 * 12),
        'pub_buffer':         (pub_fee_2022_deflated * (edu_cpi[2012]/edu_cpi[2022]) / (inc_2012*12)) -
                               pub_fee_2022_deflated / (inc_2022 * 12),
        'priv_buffer':        (prv_fee_2022_deflated * (edu_cpi[2012]/edu_cpi[2022]) / (inc_2012*12)) -
                               prv_fee_2022_deflated / (inc_2022 * 12),
    }

print(f"{'State':<25} {'Income 2022':>12} {'Growth rate':>13} {'Pub buffer':>12} {'Priv buffer':>13}")
print("-" * 80)
for state, p in state_profiles.items():
    diff = p['growth_rate'] - INCOME_GROWTH_RATE_HISTORICAL
    print(f"  {state:<23} RM {p['income_2022']:>6,}    "
          f"{p['growth_rate']*100:>9.2f}%/yr  "
          f"{p['pub_buffer']:>12.4f}  "
          f"{p['priv_buffer']:>13.4f}  "
          f"({'above' if diff>0 else 'below'} national)")


### 5b -- Generate all state scenario projections

For each state we run all four scenarios using the state's own income growth rate. Fee growth rates are the same for all states.

In [ ]:
state_projections = {}
for state, profile in state_profiles.items():
    state_projections[state] = {'public': {}, 'private': {}}
    for scenario_name, params in scenarios.items():
        scenario_income_growth = profile['growth_rate']         * params['income_mult']
        scenario_fee_growth    = EDU_CPI_GROWTH_RATE_HISTORICAL * params['fee_mult']

        state_projections[state]['public'][scenario_name] = project_affordability(
            profile['income_2022'], pub_fee_2022_deflated,
            scenario_income_growth, scenario_fee_growth, PROJ_YEARS)
        state_projections[state]['private'][scenario_name] = project_affordability(
            profile['income_2022'], prv_fee_2022_deflated,
            scenario_income_growth, scenario_fee_growth, PROJ_YEARS)

print(f"Projections generated: {len(FOCUS_STATES)} states x 4 scenarios x 2 pathways = "
      f"{len(FOCUS_STATES)*4*2} series.")
print()
print("First scenario to cause reversal per state:")
for state, profile in state_profiles.items():
    for pathway in ['public', 'private']:
        baseline = profile['pub_baseline_2012'] if pathway=='public' else profile['priv_baseline_2012']
        for scenario_name in scenarios.keys():
            proj = state_projections[state][pathway][scenario_name]
            crossed = proj[proj['affordability'] >= baseline]['year']
            if len(crossed) > 0:
                print(f"  {state:<25} {pathway:<8} -> {scenario_name.split(' -- ')[1]:<20} by {int(crossed.values[0])}")
                break
        else:
            print(f"  {state:<25} {pathway:<8} -> no reversal before 2040")


### 5c -- Compute each state's buffer and tipping year under the combined shock scenario

The combined shock (Scenario D) is the worst case. We show exactly how many years each state has before the combined shock erases all improvement.

In [ ]:
print("Years until reversal under Scenario D (Combined shock):")
print(f"{'State':<25} {'Public':>10} {'Private':>12}")
print("-" * 50)
for state, profile in state_profiles.items():
    for pathway, baseline in [
        ('public',  profile['pub_baseline_2012']),
        ('private', profile['priv_baseline_2012']),
    ]:
        proj = state_projections[state][pathway]['D -- Combined shock']
        crossed = proj[proj['affordability'] >= baseline]['year']
        if state == 'W.P. Kuala Lumpur':
            val = str(int(crossed.values[0])) if len(crossed)>0 else '>2040'
            if pathway == 'public':
                print(f"  {state:<23}", end='')
            print(f"  {val:>10}", end='')
            if pathway == 'private':
                print()
        else:
            val = str(int(crossed.values[0])) if len(crossed)>0 else '>2040'
            if pathway == 'public':
                print(f"  {state:<23}", end='')
            print(f"  {val:>10}", end='')
            if pathway == 'private':
                print()


### 5d -- Plot small multiples: six states x two pathways

Top row = public pathway. Bottom row = private pathway. Red dotted line = 2012 baseline for that state. Colour of projection lines matches the scenario colours used throughout.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(22, 9))

for col_idx, state in enumerate(FOCUS_STATES):
    profile    = state_profiles[state]
    state_data = hh_state[hh_state['state']==state].sort_values('year')

    for row_idx, (pathway, start_fee, baseline_2012, pathway_label) in enumerate([
        ('public',  pub_fee_2022_deflated, profile['pub_baseline_2012'],  'Public'),
        ('private', prv_fee_2022_deflated, profile['priv_baseline_2012'], 'Private'),
    ]):
        ax = axes[row_idx][col_idx]

        # Historical actual using deflated fees per year
        hist_aff = []
        for yr in sorted(state_data['year'].unique()):
            inc = state_data[state_data['year']==yr]['income_median'].values[0]
            fee = (pub_fee_2022_deflated if pathway=='public' else prv_fee_2022_deflated) *                   (edu_cpi[yr] / edu_cpi[2022])
            hist_aff.append({'year': yr, 'aff': fee / (inc * 12)})
        hist_df = pd.DataFrame(hist_aff)
        ax.plot(hist_df['year'], hist_df['aff'],
                color='black', lw=2, marker='o', markersize=4)

        # Four scenario projections
        for scenario_name, proj_df in state_projections[state][pathway].items():
            params = scenarios[scenario_name]
            ax.plot(proj_df['year'], proj_df['affordability'],
                    color=params['color'], lw=1.5, ls=params['ls'], alpha=0.85)

        ax.axhline(baseline_2012, color=RED, lw=1.2, ls=':', alpha=0.8)
        ax.axvline(2022, color=GRAY, lw=0.8, ls=':', alpha=0.4)
        ax.set_ylim(bottom=0)

        if col_idx == 0:
            ax.set_ylabel(f'{pathway_label}\nAffordability Index', fontsize=8)
        if row_idx == 0:
            short = state.replace('W.P. Kuala Lumpur', 'KL')
            ax.set_title(f'{short}\nRM {profile["income_2022"]:,}/mth\n'
                         f'{profile["growth_rate"]*100:.1f}%/yr growth',
                         fontsize=7.5, fontweight='bold')
        if row_idx == 1:
            ax.set_xlabel('Year', fontsize=7)

plt.suptitle('Step 5 -- Tipping Point by State: Public (top) vs Private (bottom)\n'
             'Red dotted = 2012 baseline (state-specific) | '
             'Green=Baseline  Orange=Income slowdown  Purple=Cost accel  Red=Combined shock',
             fontweight='bold', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


---
# Step 6 -- Summary Dashboard and Interpretation

This step brings together the key numbers from all five steps.
Three summary charts followed by a plain-language interpretation.

Chart 1: At the national level, which scenarios cause reversal and how soon?
Chart 2: Which states have the least buffer before their improvement reverses?
Chart 3: Which states are already growing below the income rate needed to stay safe?


### 6a -- Print the complete summary table

In [ ]:
print("=" * 70)
print("COMPLETE SUMMARY -- TIPPING POINT ANALYSIS (CPI-DEFLATED FEES)")
print("=" * 70)
print()
print("KEY GROWTH RATES:")
print(f"  Historical income growth:    {INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print(f"  Historical education CPI:    {EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print(f"  Income grew {INCOME_GROWTH_RATE_HISTORICAL/EDU_CPI_GROWTH_RATE_HISTORICAL:.1f}x faster -- this is why affordability improved")
print()
print("NATIONAL TIPPING POINT YEARS:")
print(f"{'Scenario':<30} {'Public':>14} {'Private':>14}")
print("-" * 62)
for s in scenarios.keys():
    pv = tipping_years_public.get(s)
    rv = tipping_years_private.get(s)
    print(f"  {s:<28} "
          f"{str(pv) if pv else '>2040':>14} "
          f"{str(rv) if rv else '>2040':>14}")
print()
print("SAFE ZONE THRESHOLDS:")
print(f"  Public  -- income cannot fall below: {thresholds['Public  pathway']['min_income']*100:.2f}%/yr")
print(f"  Public  -- fees cannot rise above:   {thresholds['Public  pathway']['max_fee']*100:.2f}%/yr")
print(f"  Private -- income cannot fall below: {thresholds['Private pathway']['min_income']*100:.2f}%/yr")
print(f"  Private -- fees cannot rise above:   {thresholds['Private pathway']['max_fee']*100:.2f}%/yr")
print()
print("STATE BUFFER:")
print(f"{'State':<25} {'Growth rate':>12} {'Pub buffer':>12} {'Priv buffer':>13}")
print("-" * 66)
for state, p in state_profiles.items():
    print(f"  {state:<23} {p['growth_rate']*100:>10.2f}%/yr "
          f"{p['pub_buffer']:>12.4f} {p['priv_buffer']:>13.4f}")


### 6b -- Three-chart summary dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# Chart 1: tipping point year per scenario
x = np.arange(len(scenarios))
width = 0.35
pub_tip_vals  = [tipping_years_public.get(s)  or 2042 for s in scenarios.keys()]
priv_tip_vals = [tipping_years_private.get(s) or 2042 for s in scenarios.keys()]

axes[0].bar(x - width/2, pub_tip_vals,  width, color=BLUE,   alpha=0.85, label='Public')
axes[0].bar(x + width/2, priv_tip_vals, width, color=ORANGE, alpha=0.85, label='Private')
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.split(' -- ')[1] for s in scenarios.keys()], fontsize=8.5)
axes[0].set_ylabel('Year Improvement Is Fully Erased')
axes[0].set_ylim(2020, 2046)
axes[0].axhline(2022, color=GRAY, lw=1, ls='--', alpha=0.5)
axes[0].set_title('When Does Each Scenario Erase\nthe 2012-2022 Improvement?', fontweight='bold')
axes[0].legend(fontsize=9)
for i, (pv, rv) in enumerate(zip(pub_tip_vals, priv_tip_vals)):
    axes[0].text(i-width/2, pv+0.2, str(pv) if pv<2042 else '>2040',
                 ha='center', fontsize=9, fontweight='bold', color=BLUE)
    axes[0].text(i+width/2, rv+0.2, str(rv) if rv<2042 else '>2040',
                 ha='center', fontsize=9, fontweight='bold', color=ORANGE)

# Chart 2: buffer by state
states_plot   = list(state_profiles.keys())
pub_buffers   = [state_profiles[s]['pub_buffer']  for s in states_plot]
priv_buffers  = [state_profiles[s]['priv_buffer'] for s in states_plot]
y = np.arange(len(states_plot))

axes[1].barh(y + width/2, pub_buffers,  width, color=BLUE,   alpha=0.85, label='Public')
axes[1].barh(y - width/2, priv_buffers, width, color=ORANGE, alpha=0.85, label='Private')
axes[1].set_yticks(y)
axes[1].set_yticklabels([s.replace('W.P. Kuala Lumpur','KL') for s in states_plot], fontsize=9)
axes[1].set_xlabel('Buffer (index points before reversal)')
axes[1].set_title('Buffer Per State Before\nAffordability Reverses to 2012 Level', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].axvline(0, color='black', lw=0.8)

# Chart 3: income growth rate by state
growth_rates = [state_profiles[s]['growth_rate']*100 for s in states_plot]
bar_colors_g = [GREEN if v >= INCOME_GROWTH_RATE_HISTORICAL*100 else
                ORANGE if v >= INCOME_GROWTH_RATE_HISTORICAL*50 else RED
                for v in growth_rates]
bars = axes[2].barh(states_plot, growth_rates, color=bar_colors_g, alpha=0.85)
axes[2].axvline(INCOME_GROWTH_RATE_HISTORICAL*100, color='black', lw=1.5, ls='--',
                label=f'National avg ({INCOME_GROWTH_RATE_HISTORICAL*100:.1f}%/yr)')
axes[2].set_xlabel('Historical Income Growth Rate (%/yr, 2012-2022)')
axes[2].set_title('State Income Growth Rates\n(Below national avg = more vulnerable)', fontweight='bold')
axes[2].legend(fontsize=9)
for bar, val in zip(bars, growth_rates):
    axes[2].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8.5)

plt.suptitle('Step 6 -- Summary Dashboard (CPI-Deflated Fees)',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 6c -- Final interpretation

In [ ]:
print("=" * 70)
print("INTERPRETATION")
print("=" * 70)
print()
print("WHAT DROVE THE IMPROVEMENT (Step 1):")
print(f"  From 2012 to 2022, median income grew at {INCOME_GROWTH_RATE_HISTORICAL*100:.2f}%/yr")
print(f"  while education costs only grew at {EDU_CPI_GROWTH_RATE_HISTORICAL*100:.2f}%/yr.")
print(f"  Income grew {INCOME_GROWTH_RATE_HISTORICAL/EDU_CPI_GROWTH_RATE_HISTORICAL:.1f}x faster.")
print("  Education did not get cheaper. Fees rose -- just more slowly than incomes.")
print("  Using CPI-deflated fees, the real improvement is smaller than a naive")
print("  analysis would suggest, but the direction is the same.")
print()
print("THE IMPROVEMENT IS FRAGILE (Step 2):")
print("  The 2020 COVID shock showed how quickly the index can reverse.")
print("  Income fell, fees barely moved, and both pathways worsened immediately.")
print("  The private pathway was hit harder because its fee base is 10.7x larger.")
print()
print("SAFE ZONE THRESHOLDS (Step 4):")
print(f"  Public  pathway: income must stay above {thresholds['Public  pathway']['min_income']*100:.2f}%/yr")
print(f"                   fees must stay below   {thresholds['Public  pathway']['max_fee']*100:.2f}%/yr")
print(f"  Private pathway: income must stay above {thresholds['Private pathway']['min_income']*100:.2f}%/yr")
print(f"                   fees must stay below   {thresholds['Private pathway']['max_fee']*100:.2f}%/yr")
print("  The private pathway thresholds are tighter because the fee base is larger.")
print()
most_vulnerable = min(state_profiles.items(), key=lambda x: x[1]['growth_rate'])
print("GEOGRAPHIC INEQUALITY (Step 5):")
print(f"  {most_vulnerable[0]} has the lowest income growth at")
print(f"  {most_vulnerable[1]['growth_rate']*100:.2f}%/yr and the thinnest buffer.")
print("  The same shock that KL can absorb will push low-income states")
print("  past the tipping point. A uniform national policy response")
print("  will not protect the most vulnerable states equally.")

os.makedirs('../data/analysis', exist_ok=True)
aff_national_df.to_csv('../data/analysis/affordability_tipping_deflated.csv', index=False)
print()
print("Saved -> ../data/analysis/affordability_tipping_deflated.csv")
